# SELFIES Diffusion Model Training

Discrete Denoising Diffusion Probabilistic Model (D3PM) for molecular generation using SELFIES.

## Two Training Modes:
1. **Unconditional** - Generate diverse drug-like molecules
2. **Conditional** - Generate molecules with specific properties (QED, SA, pChEMBL, etc.)

## Architecture:
- Transformer encoder with sinusoidal timestep embeddings
- Cosine noise schedule (1000 timesteps)
- Cross-entropy loss for discrete tokens

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import json
import math
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from datetime import datetime
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

import selfies as sf
from rdkit import Chem
from rdkit.Chem import QED, Descriptors

# Check GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Hyperparameters

In [ ]:
# ============================================================================
# HYPERPARAMETERS
# ============================================================================

class Config:
    # Data Paths - Compatible with both Windows and Google Colab
    # Using string paths that work cross-platform
    DATA_PATH = "/content/drive/MyDrive/Datasets/Drug Discovery/molecules_200k_selfies_clean.csv"
    TOKENIZER_PATH = "/content/drive/MyDrive/Datasets/Drug Discovery/selfies_tokenizer.json"
    MODEL_DIR = "/content/drive/MyDrive/Models"

    # Sequence
    MAX_LEN = 96              # Maximum sequence length

    # Model Architecture
    D_MODEL = 256             # Transformer hidden dimension
    N_HEADS = 8               # Number of attention heads
    N_LAYERS = 6              # Number of transformer layers
    D_FF = 1024               # Feed-forward dimension
    DROPOUT = 0.1             # Dropout rate

    # Diffusion
    TIMESTEPS = 1000          # Number of diffusion timesteps
    BETA_START = 0.0001       # Starting noise level
    BETA_END = 0.058           # Ending noise level
    GAMMA = 0.60              # Gamma Level
    SCHEDULE = "cosine"       # Noise schedule: "linear" or "cosine"

    # Training
    BATCH_SIZE = 64           # Batch size (adjust for GPU memory)
    LEARNING_RATE = 1e-4      # Learning rate
    WEIGHT_DECAY = 0.01       # AdamW weight decay
    EPOCHS = 50               # Number of training epochs
    WARMUP_STEPS = 1000       # Learning rate warmup steps
    GRAD_CLIP = 1.0           # Gradient clipping norm

    # Conditional Features (for Stage 2)
    CONDITION_FEATURES = [
        'qed',              # Drug-likeness score (0-1)
        'sa_score',         # Synthetic accessibility (1-10)
        'mw',               # Molecular weight
        'alogp',            # Lipophilicity
        'hbd',              # Hydrogen bond donors
        'hba',              # Hydrogen bond acceptors
        'psa',              # Polar surface area
        'num_rings',        # Number of rings
        'pchembl_mean',     # Binding affinity (if available)
    ]
    CONDITION_DIM = 64        # Condition embedding dimension
    COND_DROPOUT = 0.35       # ⬆️ INCREASED from 0.2 to 0.35 for stronger conditioning

    # Property Loss Settings (for QED prediction auxiliary loss)
    USE_PROPERTY_LOSS = True     # Enable QED prediction head
    PROPERTY_LOSS_WEIGHT = 1.0   # Weight for property loss

    # Data Filtering
    FILTER_MIN_QED = 0.5         # Filter to QED >= 0.5 (keeps 80.69% = 161k molecules)
                                  # Set to None to use all data

    # Sampling
    SAMPLE_STEPS = 100        # Number of sampling steps (can be < TIMESTEPS)
    TEMPERATURE = 0.7        # Sampling temperature
    CFG_SCALE = 3.0           # try 2–5

    # Logging
    LOG_EVERY = 100           # Log every N batches
    SAVE_EVERY = 5            # Save checkpoint every N epochs
    EVAL_SAMPLES = 1000       # Number of samples for evaluation

config = Config()

# Create model directory
os.makedirs(config.MODEL_DIR, exist_ok=True)

print("=" * 60)
print("HYPERPARAMETERS")
print("=" * 60)
for key, value in vars(Config).items():
    if not key.startswith('_'):
        print(f"{key}: {value}")

## Load Data & Tokenizer

In [ ]:
# Load dataset
print(f"Loading data from {config.DATA_PATH}...")
df = pd.read_csv(config.DATA_PATH)
print(f"Loaded {len(df):,} molecules")

# Load tokenizer
print(f"\nLoading tokenizer from {config.TOKENIZER_PATH}...")
with open(config.TOKENIZER_PATH, 'r') as f:
    tokenizer_data = json.load(f)

vocab = tokenizer_data['vocab']
stoi = tokenizer_data['stoi']
itos = {int(k): v for k, v in tokenizer_data['itos'].items()}
VOCAB_SIZE = len(vocab)

PAD = tokenizer_data['special_tokens']['PAD']
BOS = tokenizer_data['special_tokens']['BOS']
EOS = tokenizer_data['special_tokens']['EOS']

PAD_IDX = stoi[PAD]
BOS_IDX = stoi[BOS]
EOS_IDX = stoi[EOS]

print(f"Vocabulary size: {VOCAB_SIZE}")
print(f"Special tokens: PAD={PAD_IDX}, BOS={BOS_IDX}, EOS={EOS_IDX}")

# Show dataset columns
print(f"\nDataset columns ({len(df.columns)}):")
print(df.columns.tolist())

In [ ]:
# Check conditioning features availability
print("\n" + "="*60)
print("CONDITIONING FEATURES")
print("="*60)

# Separate features by coverage for different training strategies
high_coverage_features = []  # >50% - for main conditional training
low_coverage_features = []   # <50% but present - for specialized training (e.g., affinity)

for feat in config.CONDITION_FEATURES:
    if feat in df.columns:
        non_null = df[feat].notna().sum()
        coverage = non_null / len(df) * 100
        if coverage > 50:
            high_coverage_features.append(feat)
            print(f"✅ {feat}: {coverage:.1f}% coverage → HIGH (main conditioning)")
        else:
            low_coverage_features.append(feat)
            print(f"⚠️ {feat}: {coverage:.1f}% coverage → LOW (specialized training)")
    else:
        print(f"❌ {feat}: NOT FOUND")

print(f"\n📊 Feature Summary:")
print(f"  High coverage ({len(high_coverage_features)}): {high_coverage_features}")
print(f"  Low coverage ({len(low_coverage_features)}): {low_coverage_features}")

# Use high-coverage for main training
available_features = high_coverage_features.copy()

print(f"\n🎯 Using {len(available_features)} features for main conditioning")
print(f"💡 Low-coverage features (e.g., pchembl_mean) can be used in specialized Stage 3 training")

print("\nMissingness per conditioning feature:")
print(df[available_features].isna().mean().sort_values(ascending=False))

print("\nRows with ANY missing among available_features (%):")
print(df[available_features].isna().any(axis=1).mean() * 100)

print("\nRows with ALL present among available_features (%):")
print((~df[available_features].isna().any(axis=1)).mean() * 100)


## Tokenization Functions

In [ ]:
# NOTE: q_sample and VALID_TOKEN_IDS are defined later with diffusion functions (cell 18).
# Removed duplicate definition here to avoid confusion - run cells top-to-bottom!

def encode_selfies(selfies_str, max_len=config.MAX_LEN):
    """Encode SELFIES string to token IDs with BOS/EOS/PAD and UNK fallback."""
    tokens = list(sf.split_selfies(selfies_str))
    tokens = [BOS] + tokens + [EOS]

    # Preserve EOS when truncating
    if len(tokens) > max_len:
        tokens = tokens[:max_len]
        tokens[-1] = EOS

    pad_len = max_len - len(tokens)
    tokens += [PAD] * pad_len

    # UNK fallback (requires UNK in vocab). If you don't have UNK, replace with stoi[PAD] or raise.
    if 'UNK' in stoi:
        unk_id = stoi['UNK']
        return [stoi.get(t, unk_id) for t in tokens]
    elif '<UNK>' in stoi:
        unk_id = stoi['<UNK>']
        return [stoi.get(t, unk_id) for t in tokens]
    else:
        # Fail fast if vocab has no UNK
        missing = [t for t in tokens if t not in stoi]
        if missing:
            raise ValueError(f"Unknown tokens encountered (no UNK in vocab): {missing[:10]}")
        return [stoi[t] for t in tokens]


def decode_selfies(ids):
    """Decode token IDs back to SELFIES string."""
    tokens = [itos[i] for i in ids]
    tokens = [t for t in tokens if t not in (PAD, BOS, EOS)]
    return "".join(tokens)


def selfies_to_smiles(selfies_str):
    """Convert SELFIES to SMILES."""
    try:
        return sf.decoder(selfies_str)
    except Exception:
        return None


def is_valid_molecule(smiles):
    """Check if SMILES is a valid molecule."""
    if smiles is None:
        return False
    return Chem.MolFromSmiles(smiles) is not None


print("✅ SELFIES encoding/decoding functions defined (q_sample defined in cell 18)")

## PyTorch Dataset

In [ ]:
class MoleculeDataset(Dataset):
    """
    PyTorch Dataset for SELFIES molecules with optional conditioning.
    """
    def __init__(self, df, condition_features=None, max_len=config.MAX_LEN):
        self.selfies = df['selfies'].values
        self.max_len = max_len
        self.condition_features = condition_features

        # Prepare conditioning data
        if condition_features:
            self.conditions = df[condition_features].values.astype(np.float32)
            # Normalize conditions (z-score)
            self.cond_mean = np.nanmean(self.conditions, axis=0)
            self.cond_std = np.nanstd(self.conditions, axis=0) + 1e-8
            self.conditions = (self.conditions - self.cond_mean) / self.cond_std
            # Replace NaN with 0 (will be masked)
            self.conditions = np.nan_to_num(self.conditions, nan=0.0)
            self.has_conditions = ~np.isnan(df[condition_features].values).any(axis=1)
        else:
            self.conditions = None
            self.has_conditions = None

    def __len__(self):
        return len(self.selfies)

    def __getitem__(self, idx):
        selfies_str = self.selfies[idx]
        tokens = torch.tensor(encode_selfies(selfies_str, self.max_len), dtype=torch.long)

        if self.conditions is not None:
            cond = torch.tensor(self.conditions[idx], dtype=torch.float32)
            mask = torch.tensor(self.has_conditions[idx], dtype=torch.bool)
            return tokens, cond, mask

        return tokens


# ============================
# 📊 STAGE 1 DATA SPLIT (Simple 90-10)
# ============================
# For unconditional training, use full dataset with simple split

train_size = int(0.9 * len(df))
train_df = df.iloc[:train_size].reset_index(drop=True)
val_df = df.iloc[train_size:].reset_index(drop=True)

print(f"\n{'='*60}")
print(f"STAGE 1 DATA SPLIT (Unconditional Training)")
print(f"{'='*60}")
print(f"Train: {len(train_df):,} molecules")
print(f"Val:   {len(val_df):,} molecules")
print(f"\n💡 Note: Stage 2 (Conditional) will use filtered + stratified split")

## Diffusion Noise Schedule

In [ ]:
def get_beta_schedule(schedule_type, timesteps, beta_start, beta_end, gamma=1.0, s=0.008):
    """
    Beta schedule for DISCRETE token diffusion.
    beta_t = probability of corrupting a token at step t.
    """
    if schedule_type == "linear":
        return torch.linspace(beta_start, beta_end, timesteps)

    elif schedule_type == "cosine":
        # --- Base cosine shape (Improved DDPM) ---
        steps = timesteps + 1
        t = torch.linspace(0, timesteps, steps) / timesteps

        alphas_cumprod = torch.cos((t + s) / (1 + s) * math.pi / 2) ** 2
        alphas_cumprod = alphas_cumprod / alphas_cumprod[0]

        # Raw cosine betas (shape only)
        betas = 1 - (alphas_cumprod[1:] / alphas_cumprod[:-1])

        # Normalize to [0, 1]
        betas = (betas - betas.min()) / (betas.max() - betas.min() + 1e-8)

        # FRONT-LOAD CONTROL (gamma < 1 => more noise earlier)
        betas = betas ** gamma

        # Rescale to desired beta range
        betas = betas * (beta_end - beta_start) + beta_start

        # Safety clamp
        betas = torch.clamp(betas, min=beta_start, max=min(beta_end, 0.999))
        return betas

    else:
        raise ValueError(f"Unknown schedule: {schedule_type}")


class DiffusionSchedule:
    """
    Precompute schedule parameters.

    NOTE: For discrete token diffusion:
      keep_prob[t] = Π_{i<=t} (1 - beta_i)
    """
    def __init__(self, timesteps, schedule_type="cosine", beta_start=1e-4, beta_end=0.10, gamma=1.0):
        print("DEBUG schedule args:", timesteps, schedule_type, beta_start, beta_end, gamma)
        self.timesteps = timesteps

        # Betas (corruption probabilities)
        betas = get_beta_schedule(schedule_type, timesteps, beta_start, beta_end, gamma=gamma)

        # "Alphas" are just (1 - beta) for discrete survival
        alphas = 1 - betas
        alphas_cumprod = torch.cumprod(alphas, dim=0)
        alphas_cumprod_prev = F.pad(alphas_cumprod[:-1], (1, 0), value=1.0)

        # Store
        self.betas = betas
        self.alphas = alphas
        self.alphas_cumprod = alphas_cumprod
        self.alphas_cumprod_prev = alphas_cumprod_prev

        # Keep these if your training code expects them
        self.sqrt_alphas_cumprod = torch.sqrt(alphas_cumprod)
        self.sqrt_one_minus_alphas_cumprod = torch.sqrt(1 - alphas_cumprod)

        # Discrete keep/corrupt probabilities
        self.keep_prob = alphas_cumprod
        self.corrupt_prob = 1 - alphas_cumprod

    def to(self, device):
        self.betas = self.betas.to(device)
        self.alphas = self.alphas.to(device)
        self.alphas_cumprod = self.alphas_cumprod.to(device)
        self.alphas_cumprod_prev = self.alphas_cumprod_prev.to(device)
        self.sqrt_alphas_cumprod = self.sqrt_alphas_cumprod.to(device)
        self.sqrt_one_minus_alphas_cumprod = self.sqrt_one_minus_alphas_cumprod.to(device)
        self.keep_prob = self.keep_prob.to(device)
        self.corrupt_prob = self.corrupt_prob.to(device)
        return self


print("DEBUG config:", {
    "TIMESTEPS": config.TIMESTEPS,
    "SCHEDULE": config.SCHEDULE,
    "BETA_START": config.BETA_START,
    "BETA_END": config.BETA_END,
    "GAMMA": getattr(config, "GAMMA", None),
    "type(BETA_END)": type(config.BETA_END),
})

schedule = DiffusionSchedule(
    timesteps=config.TIMESTEPS,
    schedule_type=config.SCHEDULE,
    beta_start=config.BETA_START,
    beta_end=config.BETA_END,
    gamma=config.GAMMA
).to(device)

print(f"Noise schedule ({config.SCHEDULE}):")
print(f"  Beta range: {schedule.betas[0]:.6f} -> {schedule.betas[-1]:.6f}")
print(f"  Keep prob at t=0: {schedule.keep_prob[0]:.4f}")
print(f"  Keep prob at t=500: {schedule.keep_prob[500]:.4f}")
print(f"  Keep prob at t=999: {schedule.keep_prob[-1]:.4f}")

## Diffusion Model Architecture

In [ ]:
# ============================
# MODEL CODE (Sinusoidal + TransformerBlock + DiffusionTransformer)
# ============================

import math
import torch
import torch.nn as nn

class SinusoidalPositionEmbeddings(nn.Module):
    """Sinusoidal embeddings for timesteps."""
    def __init__(self, dim: int):
        super().__init__()
        self.dim = dim

    def forward(self, t: torch.Tensor) -> torch.Tensor:
        # ensure float for stable embeddings
        t = t.float()
        device = t.device

        half_dim = self.dim // 2
        if half_dim <= 1:
            # edge case: just return zeros if dim too small
            return torch.zeros((t.shape[0], self.dim), device=device, dtype=torch.float32)

        freq = math.log(10000.0) / (half_dim - 1)
        freq = torch.exp(torch.arange(half_dim, device=device) * -freq)  # [half_dim]
        args = t[:, None] * freq[None, :]  # [B, half_dim]
        emb = torch.cat([torch.sin(args), torch.cos(args)], dim=-1)  # [B, dim or dim-1]

        # if dim is odd, pad
        if emb.shape[-1] < self.dim:
            emb = torch.cat([emb, torch.zeros((emb.shape[0], 1), device=device)], dim=-1)
        return emb


class TransformerBlock(nn.Module):
    """Transformer encoder block."""
    def __init__(self, d_model: int, n_heads: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.drop = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, key_padding_mask: torch.Tensor | None = None) -> torch.Tensor:
        # key_padding_mask: True means "ignore this token" (padding)
        attn_out, _ = self.attn(x, x, x, key_padding_mask=key_padding_mask)
        x = self.norm1(x + self.drop(attn_out))
        x = self.norm2(x + self.ff(x))
        return x


class DiffusionTransformer(nn.Module):
    """
    Transformer denoiser for discrete sequences.
    Predicts logits over x0 tokens at each position given x_t and timestep t.
    """
    def __init__(
        self,
        vocab_size: int,
        max_len: int = 96,
        d_model: int = 256,
        n_heads: int = 8,
        n_layers: int = 6,
        d_ff: int = 1024,
        dropout: float = 0.1,
        condition_dim: int | None = None,
        n_conditions: int = 0,
    ):
        super().__init__()
        self.vocab_size = vocab_size
        self.max_len = max_len
        self.d_model = d_model
        self.conditional = (condition_dim is not None) and (n_conditions > 0)

        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_len, d_model)

        self.time_emb = nn.Sequential(
            SinusoidalPositionEmbeddings(d_model),
            nn.Linear(d_model, d_model * 2),
            nn.GELU(),
            nn.Linear(d_model * 2, d_model),
        )

        if self.conditional:
            # keep same name/shape pattern so Stage2->Stage3 weight transfer behaves as expected
            self.cond_emb = nn.Sequential(
                nn.Linear(n_conditions, condition_dim),
                nn.GELU(),
                nn.Linear(condition_dim, d_model),
            )

            # ✨ NEW: Property prediction head for QED auxiliary loss
            self.property_head = nn.Sequential(
                nn.Linear(d_model, d_model // 2),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(d_model // 2, 1),  # predict single QED value
                nn.Sigmoid()  # QED is in [0, 1]
            )
        else:
            self.property_head = None

        self.layers = nn.ModuleList([
            TransformerBlock(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)
        ])

        self.norm = nn.LayerNorm(d_model)
        self.output = nn.Linear(d_model, vocab_size)
        self.drop = nn.Dropout(dropout)

        self._init_weights()

    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def forward(self, x: torch.Tensor, t: torch.Tensor, cond: torch.Tensor | None = None, cond_mask: torch.Tensor | None = None) -> torch.Tensor:
        B, L = x.shape
        if L > self.max_len:
            raise ValueError(f"Sequence length {L} exceeds max_len={self.max_len}. Increase MAX_LEN or truncate inputs.")

        pos = torch.arange(L, device=x.device).unsqueeze(0).expand(B, -1)
        h = self.token_emb(x) + self.pos_emb(pos)

        t_emb = self.time_emb(t)  # [B, d_model]
        h = h + t_emb.unsqueeze(1)

        if self.conditional and (cond is not None):
            # CRITICAL: mask conditioning BEFORE embedding (for proper CFG)
            if cond_mask is not None:
                # If cond_mask is [B], broadcast it to [B, C]
                if cond_mask.dim() == 1:
                    cond_mask = cond_mask.unsqueeze(-1).expand_as(cond)
                cond_in = cond * cond_mask.float()
            else:
                cond_in = cond
            c_emb = self.cond_emb(cond_in)  # [B, d_model]
            h = h + c_emb.unsqueeze(1)

        h = self.drop(h)

        # True = pad => ignored by attention
        key_padding_mask = (x == PAD_IDX)

        for layer in self.layers:
            h = layer(h, key_padding_mask=key_padding_mask)

        h = self.norm(h)
        logits = self.output(h)

        # ✨ NEW: Return pooled embeddings for property prediction
        h_pooled = h.mean(dim=1)  # [B, d_model] - mean pool over sequence

        return logits, h_pooled


def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


# quick test
test_model = DiffusionTransformer(
    vocab_size=VOCAB_SIZE,
    max_len=config.MAX_LEN,
    d_model=config.D_MODEL,
    n_heads=config.N_HEADS,
    n_layers=config.N_LAYERS,
    d_ff=config.D_FF,
    dropout=config.DROPOUT
).to(device)

print(f"\nModel parameters: {count_parameters(test_model):,}")
print(f"Model size: {count_parameters(test_model) * 4 / 1e6:.1f} MB (fp32)")

x_test = torch.randint(0, VOCAB_SIZE, (2, config.MAX_LEN), device=device)
t_test = torch.randint(0, config.TIMESTEPS, (2,), device=device)
out, h_pool = test_model(x_test, t_test)
print(f"Output shape: logits={out.shape}, pooled={h_pool.shape}")

del test_model
print("✅ Model code defined (with CFG masking + property head).")

## Discrete Diffusion Functions

In [ ]:
# ============================
# DIFFUSION FUNCTIONS (q_sample / loss / sampling)
# ============================


# Precompute valid token indices (exclude special tokens from corruption sampling)
VALID_TOKEN_IDS = torch.tensor(
    [i for i in range(VOCAB_SIZE) if i not in (PAD_IDX, BOS_IDX, EOS_IDX)],
    dtype=torch.long
)
print(f"Valid tokens for corruption: {len(VALID_TOKEN_IDS)} / {VOCAB_SIZE}")


def q_sample(x_0, t, schedule, vocab_size):
    """
    Sample x_t directly from x_0 (one-shot corruption).

    Uses cumulative corruption probability:
      corrupt_prob[t] = 1 - keep_prob[t]
    and replaces corrupted positions with random VALID tokens.

    Important:
    - Never corrupt PAD positions
    - Never sample PAD/BOS/EOS as replacements
    """
    B, L = x_0.shape
    device = x_0.device

    # cumulative probability that a token is corrupted by time t
    corrupt_prob = schedule.corrupt_prob[t].unsqueeze(1).expand(B, L)  # [B, L]

    # choose which positions to corrupt
    corrupt_mask = (torch.rand(B, L, device=device) < corrupt_prob)

    # don't corrupt PAD positions
    pad_mask = (x_0 == PAD_IDX)
    corrupt_mask = corrupt_mask & (~pad_mask)

    # sample random replacement tokens (valid only)
    valid_tokens = VALID_TOKEN_IDS.to(device)
    random_indices = torch.randint(0, len(valid_tokens), (B, L), device=device)
    random_tokens = valid_tokens[random_indices]

    # apply corruption
    x_t = torch.where(corrupt_mask, random_tokens, x_0)
    return x_t


def compute_loss(model, x_0, t, schedule, vocab_size, cond=None, cond_mask=None, target_qed=None):
    """
    Cross-entropy loss: predict x_0 given x_t and timestep t.
    Ignores PAD in the target.

    ✨ NEW: Optionally adds property prediction loss for QED control.
    """
    x_t = q_sample(x_0, t, schedule, vocab_size)
    logits, h_pooled = model(x_t, t, cond, cond_mask)

    # Diffusion loss (denoising)
    diffusion_loss = F.cross_entropy(
        logits.reshape(-1, vocab_size),
        x_0.reshape(-1),
        ignore_index=PAD_IDX,
        reduction="mean"
    )

    # ✨ NEW: Property loss (QED prediction) - explicit supervision for control
    property_loss = torch.tensor(0.0, device=x_0.device)
    if model.property_head is not None and target_qed is not None:
        pred_qed = model.property_head(h_pooled).squeeze(-1)  # [B]
        property_loss = F.mse_loss(pred_qed, target_qed)

    return diffusion_loss, property_loss

@torch.no_grad()
def p_sample_step(
    model, x_t, t, schedule, vocab_size,
    temperature=1.0, cond=None, cond_mask=None,
    cfg_scale=3.0
):
    device = x_t.device
    B, L = x_t.shape

    use_cfg = (cond is not None) and (cfg_scale != 0)

    if use_cfg:
        # conditional pass: keep all condition features
        cond_mask_c = cond_mask

        # unconditional pass: mask ALL condition features [B, C]
        cond_mask_u = torch.zeros_like(cond, dtype=torch.bool)
        cond_u = torch.zeros_like(cond)  # zeroed condition (masked out anyway)

        logits_u, _ = model(x_t, t, cond_u, cond_mask_u)
        logits_c, _ = model(x_t, t, cond, cond_mask_c)

        logits = logits_u + cfg_scale * (logits_c - logits_u)

        # ✨ NEW: CFG Rescale - stabilize logits magnitude (prevents explosion at high CFG)
        std_u = logits_u.std(dim=-1, keepdim=True)
        std_combined = logits.std(dim=-1, keepdim=True)
        logits = logits * (std_u / (std_combined + 1e-8))

        # sanity check: logits should diverge
        delta = (logits_c - logits_u).abs().mean().item()
    else:
        logits, _ = model(x_t, t, cond, cond_mask)
        delta = None

    # ---- debug only on first denoise step ----
    t_val = int(t[0].item())
    if t_val == schedule.timesteps - 1:
        print("cfg_scale", cfg_scale)
        print("use_cfg", use_cfg)
        print("cond is None?", cond is None)
        print("cond_mask mean", cond_mask.float().mean().item() if cond_mask is not None else None)
        if use_cfg:
            print("cond_mask_u mean", cond_mask_u.float().mean().item())
            print("mean |logits_c - logits_u| =", delta)

    # CRITICAL FIX: Mask invalid tokens (PAD/BOS/EOS) before sampling
    # This prevents collapse to trivial sequences
    logits[:, :, PAD_IDX] = -1e9
    logits[:, :, BOS_IDX] = -1e9
    logits[:, :, EOS_IDX] = -1e9  # block EOS from being sampled (model learns where to place it)

    logits = logits / max(float(temperature), 1e-8)
    probs = torch.softmax(logits, dim=-1)

    x0_pred = torch.multinomial(probs.view(-1, vocab_size), 1).view(B, L)

    if t_val <= 0:
        x0_pred[:, 0] = BOS_IDX
        return torch.where(x_t == PAD_IDX, x_t, x0_pred)

    beta_t = float(schedule.betas[t_val])
    update_mask = (torch.rand(B, L, device=device) < beta_t)
    update_mask &= (x_t != PAD_IDX)
    update_mask[:, 0] = False

    x_prev = torch.where(update_mask, x0_pred, x_t)
    x_prev[:, 0] = BOS_IDX
    return x_prev




@torch.no_grad()
def sample(
    model,
    schedule,
    num_samples,
    seq_len,
    vocab_size,
    sample_steps=None,
    temperature=1.0,
    cond=None,
    cond_mask=None,
    device="cuda"
):
    """
    Generate sequences by iterative denoising.

    Starts from uniform random VALID tokens (no PAD/BOS/EOS),
    then fixes BOS at position 0.
    """
    model.eval()

    if sample_steps is None:
        sample_steps = schedule.timesteps

    # start from random valid tokens
    valid_tokens = VALID_TOKEN_IDS.to(device)
    x = valid_tokens[torch.randint(0, len(valid_tokens), (num_samples, seq_len), device=device)]
    x[:, 0] = BOS_IDX  # enforce BOS

    # choose timesteps to traverse (skip schedule)
    timesteps = torch.linspace(schedule.timesteps - 1, 0, sample_steps, dtype=torch.long, device=device)

    for t in tqdm(timesteps, desc="Sampling"):
        t_batch = t.expand(num_samples)
        cfg_scale = getattr(config, "CFG_SCALE", 3.0)

        x = p_sample_step(
            model, x, t_batch, schedule, vocab_size,
            temperature=temperature,
            cond=cond,
            cond_mask=cond_mask,
            cfg_scale=cfg_scale
        )

    x[:, 0] = BOS_IDX
    return x


print("✅ Diffusion functions defined (with CFG fixes + logit masking + property loss + CFG rescale).")

---
# Stage 1: Unconditional Training
---

In [ ]:
# Create unconditional dataset and dataloader
train_dataset_uncond = MoleculeDataset(train_df, condition_features=None)
val_dataset_uncond = MoleculeDataset(val_df, condition_features=None)

train_loader_uncond = DataLoader(
    train_dataset_uncond,
    batch_size=config.BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)
val_loader_uncond = DataLoader(
    val_dataset_uncond,
    batch_size=config.BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

print(f"Train batches: {len(train_loader_uncond)}")
print(f"Val batches: {len(val_loader_uncond)}")

In [ ]:
# Create unconditional model
model_uncond = DiffusionTransformer(
    vocab_size=VOCAB_SIZE,
    max_len=config.MAX_LEN,
    d_model=config.D_MODEL,
    n_heads=config.N_HEADS,
    n_layers=config.N_LAYERS,
    d_ff=config.D_FF,
    dropout=config.DROPOUT,
    condition_dim=None,
    n_conditions=0
).to(device)

print(f"Unconditional model parameters: {count_parameters(model_uncond):,}")

# Optimizer and scheduler
optimizer_uncond = AdamW(
    model_uncond.parameters(),
    lr=config.LEARNING_RATE,
    weight_decay=config.WEIGHT_DECAY
)

total_steps = len(train_loader_uncond) * config.EPOCHS
scheduler_uncond = CosineAnnealingLR(optimizer_uncond, T_max=total_steps)

In [ ]:
def train_epoch(model, loader, optimizer, scheduler, schedule, vocab_size, epoch, conditional=False):
    model.train()
    total_loss = 0.0
    num_batches = 0

    pbar = tqdm(loader, desc=f"Epoch {epoch+1}")
    for batch_idx, batch in enumerate(pbar):

        # ---- unpack batch ----
        if conditional:
            x, cond, cond_mask = batch
            x = x.to(device)
            cond = cond.to(device)
            cond_mask = cond_mask.to(device)
        else:
            x = batch.to(device)
            cond = None
            cond_mask = None

        # ---- sample timestep ----
        t = torch.randint(
            0, schedule.timesteps, (x.shape[0],), device=device
        )

        # ---- forward + backward ----
        optimizer.zero_grad(set_to_none=True)

        loss = compute_loss(
            model, x, t, schedule, vocab_size, cond, cond_mask
        )

        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            model.parameters(), config.GRAD_CLIP
        )

        optimizer.step()
        scheduler.step()

        # ---- logging ----
        total_loss += loss.item()
        num_batches += 1

        if batch_idx % config.LOG_EVERY == 0:
            pbar.set_postfix({
                "loss": f"{loss.item():.4f}",
                "lr": f"{scheduler.get_last_lr()[0]:.2e}"
            })

    return total_loss / num_batches


@torch.no_grad()
def validate(model, loader, schedule, vocab_size, conditional=False):
    model.eval()
    total_loss = 0.0
    num_batches = 0

    for batch_idx, batch in enumerate(loader):

        if conditional:
            x, cond, cond_mask = batch
            x = x.to(device)
            cond = cond.to(device)
            cond_mask = cond_mask.to(device)
        else:
            x = batch.to(device)
            cond = None
            cond_mask = None

        # ---- FIXED timestep for stability ----
        t = torch.full(
            (x.shape[0],),
            schedule.timesteps // 2,
            device=device,
            dtype=torch.long
        )

        loss_output = compute_loss(
            model, x, t, schedule, vocab_size, cond, cond_mask
        )

        # Handle tuple return from compute_loss (diffusion_loss, property_loss)
        if isinstance(loss_output, tuple):
            diffusion_loss, property_loss = loss_output
            # Combine with same weight as training
            loss = diffusion_loss + config.PROPERTY_LOSS_WEIGHT * property_loss
        else:
            loss = loss_output

        total_loss += loss.item()
        num_batches += 1

    return total_loss / num_batches


In [ ]:
os.makedirs(config.MODEL_DIR, exist_ok=True)

print("="*60)
print("STAGE 1: UNCONDITIONAL TRAINING")
print("="*60)

train_losses_uncond = []
val_losses_uncond = []
best_val_loss = float('inf')

# Optional: resume
start_epoch = 0
resume_path = getattr(config, "RESUME_PATH", None)

if resume_path and os.path.exists(resume_path):
    ckpt = torch.load(resume_path, map_location=device)

    model_uncond.load_state_dict(ckpt["model_state_dict"])
    optimizer_uncond.load_state_dict(ckpt["optimizer_state_dict"])

    if ckpt.get("scheduler_state_dict") is not None:
        scheduler_uncond.load_state_dict(ckpt["scheduler_state_dict"])

    train_losses_uncond = ckpt.get("train_losses", [])
    val_losses_uncond = ckpt.get("val_losses", [])
    best_val_loss = ckpt.get("best_val_loss", best_val_loss)
    start_epoch = ckpt.get("epoch", 0)

    print(f"✅ Resumed from {resume_path} at epoch {start_epoch}")

for epoch in range(start_epoch, config.EPOCHS):
    train_loss = train_epoch(
        model_uncond,
        train_loader_uncond,
        optimizer_uncond,
        scheduler_uncond,
        schedule,
        VOCAB_SIZE,
        epoch,
        conditional=False
    )
    train_losses_uncond.append(train_loss)

    val_loss = validate(
        model_uncond,
        val_loader_uncond,
        schedule,
        VOCAB_SIZE,
        conditional=False
    )
    val_losses_uncond.append(val_loss)

    print(f"Epoch {epoch+1}/{config.EPOCHS} - Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")

    # ---- save best (update best_val_loss BEFORE saving latest) ----
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({
            "epoch": epoch + 1,
            "model_state_dict": model_uncond.state_dict(),
            "optimizer_state_dict": optimizer_uncond.state_dict(),
            "scheduler_state_dict": scheduler_uncond.state_dict(),
            "train_loss": train_loss,
            "val_loss": val_loss,
            "best_val_loss": best_val_loss,
            "config": vars(config),
        }, os.path.join(config.MODEL_DIR, "diffusion_uncond_best.pt"))
        print(f"  ✅ Saved best model (val_loss: {val_loss:.4f})")

    # ---- always save latest (so you can resume anytime) ----
    torch.save({
        "epoch": epoch + 1,  # next epoch to run
        "model_state_dict": model_uncond.state_dict(),
        "optimizer_state_dict": optimizer_uncond.state_dict(),
        "scheduler_state_dict": scheduler_uncond.state_dict(),
        "train_losses": train_losses_uncond,
        "val_losses": val_losses_uncond,
        "best_val_loss": best_val_loss,
        "config": vars(config),
    }, os.path.join(config.MODEL_DIR, "diffusion_uncond_latest.pt"))

    # ---- periodic checkpoint ----
    if (epoch + 1) % config.SAVE_EVERY == 0:
        torch.save({
            "epoch": epoch + 1,
            "model_state_dict": model_uncond.state_dict(),
            "optimizer_state_dict": optimizer_uncond.state_dict(),
            "scheduler_state_dict": scheduler_uncond.state_dict(),
            "train_losses": train_losses_uncond,
            "val_losses": val_losses_uncond,
            "best_val_loss": best_val_loss,
        }, os.path.join(config.MODEL_DIR, f"diffusion_uncond_epoch{epoch+1}.pt"))

print("\n✅ Unconditional training complete!")


## Sample from Unconditional Model

In [ ]:
# Load best model
checkpoint = torch.load(os.path.join(config.MODEL_DIR, 'diffusion_uncond_best.pt'))
model_uncond.load_state_dict(checkpoint['model_state_dict'])
print(f"Loaded best model from epoch {checkpoint['epoch']}")
model_uncond.eval()


# Generate samples
print("\nGenerating 100 molecules...")
with torch.no_grad():
  samples = sample(
      model_uncond, schedule,
      num_samples=100,
      seq_len=config.MAX_LEN,
      vocab_size=VOCAB_SIZE,
      sample_steps=config.SAMPLE_STEPS,
      temperature=config.TEMPERATURE,
      device=device
  )

# Decode and validate
generated_selfies = [decode_selfies(s.cpu().tolist()) for s in samples]
generated_smiles = [selfies_to_smiles(s) for s in generated_selfies]
valid_smiles = [s for s in generated_smiles if is_valid_molecule(s)]

validity = len(valid_smiles) / len(generated_smiles) * 100
uniqueness = len(set(valid_smiles)) / len(valid_smiles) * 100 if valid_smiles else 0

print(f"\n=== Unconditional Generation Results ===")
print(f"Validity: {validity:.1f}%")
print(f"Uniqueness: {uniqueness:.1f}%")
print(f"\nSample molecules:")
for i, smi in enumerate(valid_smiles[:5]):
    mol = Chem.MolFromSmiles(smi)
    qed_score = QED.qed(mol)
    print(f"  {i+1}. {smi[:60]}... (QED: {qed_score:.3f})")

---
# Stage 2: Conditional Training
---

In [ ]:
# ============================
# 📊 STAGE 2 DATA PREPARATION
# ============================
# For conditional training, apply QED filtering + stratified split

print("\n" + "="*60)
print("STAGE 2: DATA FILTERING & STRATIFIED SPLIT")
print("="*60)

# Option 1: Filter to QED >= 0.5 (focus on drug-like molecules)
FILTER_MIN_QED = config.FILTER_MIN_QED  # From Config: 0.5 or None

if FILTER_MIN_QED is not None:
    df_filtered = df[df['qed'] >= FILTER_MIN_QED].reset_index(drop=True)
    print(f"\n🎯 FILTERED TO QED >= {FILTER_MIN_QED}")
    print(f"Original: {len(df):,} molecules")
    print(f"Filtered: {len(df_filtered):,} molecules ({len(df_filtered)/len(df)*100:.1f}%)")
    print(f"QED range: [{df_filtered['qed'].min():.3f}, {df_filtered['qed'].max():.3f}]")
    df_stage2 = df_filtered
else:
    df_stage2 = df
    print(f"\n✓ Using full dataset: {len(df):,} molecules")

# Option 2: Stratified split by QED quartiles (ensures balanced distribution)
from sklearn.model_selection import train_test_split

# Create QED bins for stratification
df_stage2['qed_bin'] = pd.cut(df_stage2['qed'], bins=4, labels=['Q1', 'Q2', 'Q3', 'Q4'])

# Stratified 90-10 split
train_df_stage2, val_df_stage2 = train_test_split(
    df_stage2,
    test_size=0.1,
    stratify=df_stage2['qed_bin'],
    random_state=42
)

train_df_stage2 = train_df_stage2.reset_index(drop=True)
val_df_stage2 = val_df_stage2.reset_index(drop=True)

# Show distributions
print(f"\n✅ STRATIFIED SPLIT (balanced QED distribution)")
print(f"Train: {len(train_df_stage2):,} molecules")
print(f"Val:   {len(val_df_stage2):,} molecules")

print(f"\nQED Distribution Check:")
print(f"Train QED: mean={train_df_stage2['qed'].mean():.3f}, median={train_df_stage2['qed'].median():.3f}")
print(f"Val QED:   mean={val_df_stage2['qed'].mean():.3f}, median={val_df_stage2['qed'].median():.3f}")

print(f"\nHigh-QED (≥0.85) representation:")
train_high = (train_df_stage2['qed'] >= 0.85).sum()
val_high = (val_df_stage2['qed'] >= 0.85).sum()
print(f"Train: {train_high:,} ({train_high/len(train_df_stage2)*100:.2f}%)")
print(f"Val:   {val_high:,} ({val_high/len(val_df_stage2)*100:.2f}%)")

# Drop temporary column
train_df_stage2 = train_df_stage2.drop(columns=['qed_bin'])
val_df_stage2 = val_df_stage2.drop(columns=['qed_bin'])

print("\n" + "="*60)

# Create conditional dataset with available features
print(f"\nUsing conditioning features: {available_features}")

train_dataset_cond = MoleculeDataset(train_df_stage2, condition_features=available_features)
val_dataset_cond = MoleculeDataset(val_df_stage2, condition_features=available_features)

assert np.all(train_dataset_cond.cond_std > 0), \
    "Some conditioning features have zero std. Check data preprocessing."

train_loader_cond = DataLoader(
    train_dataset_cond,
    batch_size=config.BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

val_loader_cond = DataLoader(
    val_dataset_cond,
    batch_size=config.BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

print(f"Train batches: {len(train_loader_cond)}")
print(f"Val batches: {len(val_loader_cond)}")

# Save normalization stats for inference
cond_stats = {
    'features': available_features,
    'mean': train_dataset_cond.cond_mean.tolist(),
    'std': train_dataset_cond.cond_std.tolist()
}

with open(os.path.join(config.MODEL_DIR, 'condition_stats.json'), 'w') as f:
    json.dump(cond_stats, f, indent=2)

print(f"\nCondition normalization stats saved.")


In [ ]:
# Create conditional model
model_cond = DiffusionTransformer(
    vocab_size=VOCAB_SIZE,
    max_len=config.MAX_LEN,
    d_model=config.D_MODEL,
    n_heads=config.N_HEADS,
    n_layers=config.N_LAYERS,
    d_ff=config.D_FF,
    dropout=config.DROPOUT,
    condition_dim=config.CONDITION_DIM,
    n_conditions=len(available_features)
).to(device)

# ✅ Load Stage-1 weights into Stage-2 model (do this HERE)
ckpt = torch.load(os.path.join(config.MODEL_DIR, "diffusion_uncond_best.pt"), map_location=device)
missing, unexpected = model_cond.load_state_dict(ckpt["model_state_dict"], strict=False)
print("Loaded uncond weights into conditional model.")
print("Missing keys (expected condition modules):", len(missing))
print("Unexpected keys:", len(unexpected))

# Now create optimizer/scheduler AFTER loading weights
optimizer_cond = AdamW(model_cond.parameters(), lr=config.LEARNING_RATE, weight_decay=config.WEIGHT_DECAY)
total_steps = len(train_loader_cond) * config.EPOCHS
scheduler_cond = CosineAnnealingLR(optimizer_cond, T_max=total_steps)


In [ ]:
import os
import torch

os.makedirs(config.MODEL_DIR, exist_ok=True)

# ============================================================================
# TRAIN CONDITIONAL MODEL (with CFG dropout + property loss)
# ============================================================================

print("="*60)
print("STAGE 2: CONDITIONAL TRAINING")
print(f"CFG dropout: {config.COND_DROPOUT}")
print(f"Property loss: {config.USE_PROPERTY_LOSS}")
print("="*60)

train_losses_cond = []
val_losses_cond = []
best_val_loss_cond = float("inf")

# Optional: resume
start_epoch = 0
resume_path = getattr(config, "RESUME_PATH_COND", None)  # set to ".../diffusion_cond_latest.pt" to resume

if resume_path and os.path.exists(resume_path):
    ckpt = torch.load(resume_path, map_location=device)

    model_cond.load_state_dict(ckpt["model_state_dict"])
    optimizer_cond.load_state_dict(ckpt["optimizer_state_dict"])

    if ckpt.get("scheduler_state_dict") is not None:
        scheduler_cond.load_state_dict(ckpt["scheduler_state_dict"])

    train_losses_cond = ckpt.get("train_losses", [])
    val_losses_cond = ckpt.get("val_losses", [])
    best_val_loss_cond = ckpt.get("best_val_loss", best_val_loss_cond)
    start_epoch = ckpt.get("epoch", 0)  # NOTE: epoch is "next epoch to run"

    saved_feats = ckpt.get("condition_features", None)
    if saved_feats is not None and list(saved_feats) != list(available_features):
        raise ValueError(f"Condition feature mismatch!\nSaved: {saved_feats}\nCurrent: {available_features}")

    print(f"✅ Resumed conditional training from {resume_path} at epoch {start_epoch}")

# ------------------------
# Wrap train_epoch to inject CFG dropout + property loss
# ------------------------
def train_epoch_cond_with_dropout(model, loader, optimizer, scheduler, schedule, vocab_size, epoch, dataset):
    model.train()
    total_diffusion_loss = 0.0
    total_property_loss = 0.0
    total_combined_loss = 0.0
    num_batches = 0

    p_drop = float(getattr(config, "COND_DROPOUT", 0.35))
    use_property_loss = getattr(config, "USE_PROPERTY_LOSS", True)
    property_weight = getattr(config, "PROPERTY_LOSS_WEIGHT", 0.1)

    # Get QED index in features
    qed_idx = available_features.index('qed') if 'qed' in available_features else None

    pbar = tqdm(loader, desc=f"Epoch {epoch+1}")
    for batch_idx, batch in enumerate(pbar):
        x, cond, cond_mask = batch
        x = x.to(device)
        cond = cond.to(device)
        cond_mask = cond_mask.to(device)  # expected shape: [B] or [B, C]

        # --- CFG training trick: randomly drop conditioning per-sample ---
        if p_drop > 0:
            drop = (torch.rand(x.shape[0], device=device) < p_drop)  # [B]
            cond_mask = cond_mask.clone()
            # Handle both [B] and [B, C] mask shapes
            if cond_mask.dim() == 1:
                cond_mask[drop] = 0  # [B] mask
            else:
                cond_mask[drop, :] = 0  # [B, C] mask - zero all features for dropped samples

        # ✨ NEW: Extract target QED for property loss
        target_qed = None
        if use_property_loss and qed_idx is not None:
            # Denormalize QED from condition vector
            cond_stats_mean = dataset.cond_mean[qed_idx]
            cond_stats_std = dataset.cond_std[qed_idx]
            target_qed = cond[:, qed_idx] * cond_stats_std + cond_stats_mean  # [B]
            target_qed = target_qed.to(device)

        t = torch.randint(0, schedule.timesteps, (x.shape[0],), device=device)

        # ✨ NEW: compute_loss now returns (diffusion_loss, property_loss)
        diffusion_loss, property_loss = compute_loss(
            model, x, t, schedule, vocab_size, cond, cond_mask, target_qed
        )

        # Combined loss
        loss = diffusion_loss + property_weight * property_loss

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), config.GRAD_CLIP)
        optimizer.step()
        scheduler.step()

        total_diffusion_loss += diffusion_loss.item()
        total_property_loss += property_loss.item()
        total_combined_loss += loss.item()
        num_batches += 1

        if batch_idx % config.LOG_EVERY == 0:
            pbar.set_postfix({
                "d_loss": f"{diffusion_loss.item():.4f}",
                "p_loss": f"{property_loss.item():.4f}",
                "total": f"{loss.item():.4f}",
                "lr": f"{scheduler.get_last_lr()[0]:.2e}"
            })

    return (
        total_combined_loss / max(1, num_batches),
        total_diffusion_loss / max(1, num_batches),
        total_property_loss / max(1, num_batches)
    )


for epoch in range(start_epoch, config.EPOCHS):
    # Train (conditional=True + dropout + property loss)
    train_loss, train_diff_loss, train_prop_loss = train_epoch_cond_with_dropout(
        model_cond, train_loader_cond, optimizer_cond, scheduler_cond,
        schedule, VOCAB_SIZE, epoch, train_dataset_cond
    )
    train_losses_cond.append(train_loss)

    # Validate (no dropout in val)
    val_loss = validate(model_cond, val_loader_cond, schedule, VOCAB_SIZE, conditional=True)
    val_losses_cond.append(val_loss)

    print(f"Epoch {epoch+1}/{config.EPOCHS} - "
          f"Train: {train_loss:.4f} (diff={train_diff_loss:.4f}, prop={train_prop_loss:.4f}), "
          f"Val: {val_loss:.4f}")

    # ---- save best ----
    if val_loss < best_val_loss_cond:
        best_val_loss_cond = val_loss
        torch.save({
            "epoch": epoch + 1,  # next epoch to run
            "model_state_dict": model_cond.state_dict(),
            "optimizer_state_dict": optimizer_cond.state_dict(),
            "scheduler_state_dict": scheduler_cond.state_dict(),
            "train_loss": train_loss,
            "val_loss": val_loss,
            "best_val_loss": best_val_loss_cond,
            "config": vars(config),
            "condition_features": available_features,
        }, os.path.join(config.MODEL_DIR, "diffusion_cond_best.pt"))
        print(f"  ✅ Saved best model (val_loss: {val_loss:.4f})")

    # ---- always save latest ----
    torch.save({
        "epoch": epoch + 1,  # next epoch to run
        "model_state_dict": model_cond.state_dict(),
        "optimizer_state_dict": optimizer_cond.state_dict(),
        "scheduler_state_dict": scheduler_cond.state_dict(),
        "train_losses": train_losses_cond,
        "val_losses": val_losses_cond,
        "best_val_loss": best_val_loss_cond,
        "config": vars(config),
        "condition_features": available_features,
    }, os.path.join(config.MODEL_DIR, "diffusion_cond_latest.pt"))

    # ---- periodic checkpoint ----
    if (epoch + 1) % config.SAVE_EVERY == 0:
        torch.save({
            "epoch": epoch + 1,
            "model_state_dict": model_cond.state_dict(),
            "optimizer_state_dict": optimizer_cond.state_dict(),
            "scheduler_state_dict": scheduler_cond.state_dict(),
            "train_losses": train_losses_cond,
            "val_losses": val_losses_cond,
            "best_val_loss": best_val_loss_cond,
            "condition_features": available_features,
        }, os.path.join(config.MODEL_DIR, f"diffusion_cond_epoch{epoch+1}.pt"))

print("\n✅ Conditional training complete!")

### ✨ NEW IMPROVEMENTS FOR STRONGER CONDITIONAL CONTROL

**Three major upgrades applied:**

#### 1. **Increased COND_DROPOUT: 0.2 → 0.35**
- Forces model to rely more on conditioning during training
- Improves CFG effectiveness at inference time
- Better generalization to unseen condition values

#### 2. **Added Property Loss (QED Prediction Head)**
- New `property_head` in `DiffusionTransformer`
- Predicts QED from pooled embeddings
- Explicit supervision: `MSE(predicted_QED, target_QED)`
- Weight: 0.1 × diffusion_loss
- **Direct signal for QED control** (not just denoising)

#### 3. **Added CFG Rescale**
- Stabilizes logits magnitude at high CFG scales
- Prevents mode collapse and artifacts
- Formula: `logits * (std_u / std_combined)`
- Maintains diversity while enabling stronger guidance

**Expected Results:**
- 🎯 **Better QED control**: Target 0.85 → actual ~0.75+ (not 0.37)
- 🎲 **Higher uniqueness**: 50-70% at all targets (not 2.5% at QED=0.5)
- 📊 **Stable across CFG scales**: Works well from 1.5 to 7.0

**Training logs now show:**
- `d_loss`: Diffusion (denoising) loss
- `p_loss`: Property (QED prediction) loss
- `total`: Combined loss

---

## Sample from Conditional Model

In [ ]:
# ===========================
# Cell 2: Helpers (strip + condition prep)
# ===========================
def strip_isotopes(smiles):
    m = Chem.MolFromSmiles(smiles)
    if not m:
        return None
    for a in m.GetAtoms():
        a.SetIsotope(0)
    return Chem.MolToSmiles(m)

def prepare_conditions(target_values, cond_stats, device, clamp_sigma=3.0):
    features = cond_stats["features"]
    mean = np.array(cond_stats["mean"], dtype=np.float32)
    std  = np.array(cond_stats["std"],  dtype=np.float32)

    values = []
    for i, feat in enumerate(features):
        if feat in target_values:
            val = float(target_values[feat])
            min_val = mean[i] - clamp_sigma * std[i]
            max_val = mean[i] + clamp_sigma * std[i]
            val_clamped = float(np.clip(val, min_val, max_val))
            if val != val_clamped:
                print(f"  ⚠️  {feat}: {val:.2f} clamped to [{min_val:.2f}, {max_val:.2f}] → {val_clamped:.2f}")
            values.append(val_clamped)
        else:
            values.append(float(mean[i]))

    values = np.array(values, dtype=np.float32)
    values_norm = (values - mean) / std
    return torch.tensor(values_norm, dtype=torch.float32, device=device)  # [n_cond]


In [ ]:
# ===========================
# Cell 3: Helpers (decode + sampling)
# ===========================
def decode_samples_to_smiles(samples):
    selfies_list = [decode_selfies(s.cpu().tolist()) for s in samples]
    smiles_list  = [selfies_to_smiles(sf) for sf in selfies_list]
    smiles_list  = [strip_isotopes(s) if s else None for s in smiles_list]
    smiles_list  = [s for s in smiles_list if s and is_valid_molecule(s)]
    return smiles_list

def sample_conditional(target, n=200, steps=300, temperature=0.7):
    cond_vec   = prepare_conditions(target, cond_stats, device)                 # [n_cond]
    cond_batch = cond_vec.unsqueeze(0).expand(n, -1).contiguous()              # [B, n_cond]
    cond_mask  = torch.ones(n, dtype=torch.bool, device=device)                # [B]  IMPORTANT

    model_cond.eval()
    with torch.no_grad():
        samples = sample(
            model_cond, schedule,
            num_samples=n,
            seq_len=config.MAX_LEN,
            vocab_size=VOCAB_SIZE,
            sample_steps=steps,
            temperature=temperature,
            cond=cond_batch,
            cond_mask=cond_mask,
            device=device
        )
    return decode_samples_to_smiles(samples)

def eval_target_qed(qed_target, sa_target=3.0, n=200, steps=300, temperature=0.7):
    target = {"qed": float(qed_target), "sa_score": float(sa_target)}
    smiles = sample_conditional(target, n=n, steps=steps, temperature=temperature)

    qeds = []
    for s in smiles:
        m = Chem.MolFromSmiles(s)
        if m:
            qeds.append(QED.qed(m))

    uniq = (len(set(smiles)) / len(smiles) * 100) if smiles else 0
    mean_qed = float(np.mean(qeds)) if qeds else None
    return mean_qed, uniq, len(smiles)



In [ ]:
# ===========================
# Cell 4: Load model + condition stats
# ===========================
ckpt_path = os.path.join(config.MODEL_DIR, "diffusion_cond_best.pt")
checkpoint = torch.load(ckpt_path, map_location=device)

model_cond.load_state_dict(checkpoint["model_state_dict"])
model_cond.eval()
print(f"Loaded best conditional model (trained up to epoch {checkpoint['epoch']})")

with open(os.path.join(config.MODEL_DIR, "condition_stats.json"), "r") as f:
    cond_stats = json.load(f)

print("Conditioning features:", cond_stats["features"])
assert np.all(np.array(cond_stats["std"], dtype=np.float32) > 0), "Some conditioning stds are zero!"



In [ ]:
# ===========================
# Verification: Ensure we have the right model and checkpoint
# ===========================
print("="*60)
print("VERIFICATION CHECKS")
print("="*60)
print(f"✅ Checkpoint path: {ckpt_path}")
print(f"✅ Model has cond_emb: {hasattr(model_cond, 'cond_emb')}")
print(f"✅ Checkpoint is conditional: {'condition_features' in checkpoint}")

if 'condition_features' in checkpoint:
    print(f"✅ Checkpoint features: {checkpoint['condition_features']}")
    assert checkpoint['condition_features'] == cond_stats['features'], \
        "❌ Feature mismatch between checkpoint and stats!"

print(f"✅ CFG scale: {getattr(config, 'CFG_SCALE', 3.0)}")
print("="*60)
print("✅ All checks passed. Ready to sample with CFG!")
print("="*60)

---

## 🎯 Next Steps: You Have Two Options

### **Option A: Quick Test (CFG Tuning Only)** ⚡
Run the CFG sweep (cell 38) with your **current model**:
- Takes 5-10 minutes
- No retraining needed
- Tests if different CFG scales help
- **Do this first** to see baseline improvement

### **Option B: Full Retraining (All Improvements)** 🚀

**After Option A, if control is still weak:**

1. **Restart kernel** (critical!)
2. **Run cells 1-29** (up to training loop)
3. **Train Stage 2** with new improvements:
   - COND_DROPOUT = 0.35 (was 0.2)
   - Property loss enabled
   - CFG rescale active
4. **Sample and test** - should see much better control

**Training will show 3 losses:**
```
d_loss: 2.3456  # diffusion (denoising) loss
p_loss: 0.0123  # property (QED prediction) loss
total: 2.3580   # combined loss
```

### 🎯 My Recommendation:

1. **Run CFG sweep first** (cell 38) - 5 min, no risk
2. **If QED=0.5 uniqueness > 40% with lower CFG** → just use adaptive CFG, no retrain needed
3. **If still weak** → retrain with all 3 improvements

---

### 📝 Implementation Complete!

All code changes applied:
- ✅ `config.COND_DROPOUT = 0.35`
- ✅ `DiffusionTransformer` has `property_head`
- ✅ `compute_loss()` returns `(diffusion_loss, property_loss)`
- ✅ `p_sample_step()` has CFG rescale
- ✅ Training loop uses property loss

**Ready to train!** 🎉

---

### 🔬 CFG Scale Sweep (Diagnose Control vs. Diversity Tradeoff)

Test different CFG scales to find the sweet spot:
- **Lower CFG (1.5-2.0)**: More diversity, weaker control
- **Higher CFG (5.0+)**: Stronger control, risk of mode collapse

In [ ]:
# Test CFG scales: 1.5, 2.5, 3.0, 5.0, 7.0
cfg_scales = [1.5, 2.5, 3.0, 5.0, 7.0]
test_targets = [0.2, 0.5, 0.85]

results = []

# Temporarily modify sample function to accept custom cfg_scale
def sample_with_cfg_scale(target_qed, cfg_scale, n=100, steps=300):
    target = {"qed": float(target_qed), "sa_score": 3.0}
    cond_vec = prepare_conditions(target, cond_stats, device)
    cond_batch = cond_vec.unsqueeze(0).expand(n, -1).contiguous()
    cond_mask = torch.ones(n, dtype=torch.bool, device=device)

    # Temporarily override config CFG_SCALE
    original_cfg = getattr(config, 'CFG_SCALE', 3.0)
    config.CFG_SCALE = cfg_scale

    model_cond.eval()
    with torch.no_grad():
        samples = sample(
            model_cond, schedule,
            num_samples=n,
            seq_len=config.MAX_LEN,
            vocab_size=VOCAB_SIZE,
            sample_steps=steps,
            temperature=0.7,
            cond=cond_batch,
            cond_mask=cond_mask,
            device=device
        )

    # Restore original CFG
    config.CFG_SCALE = original_cfg

    smiles = decode_samples_to_smiles(samples)
    qeds = [QED.qed(Chem.MolFromSmiles(s)) for s in smiles if Chem.MolFromSmiles(s)]

    return {
        'mean_qed': float(np.mean(qeds)) if qeds else 0,
        'uniqueness': len(set(smiles)) / len(smiles) * 100 if smiles else 0,
        'valid': len(smiles)
    }

print("="*70)
print("CFG SCALE SWEEP")
print("="*70)

for cfg in cfg_scales:
    print(f"\n📊 CFG Scale = {cfg}")
    print("-" * 70)
    for target_qed in test_targets:
        result = sample_with_cfg_scale(target_qed, cfg, n=100, steps=300)
        results.append({
            'cfg_scale': cfg,
            'target_qed': target_qed,
            **result
        })
        print(f"  Target QED={target_qed}: "
              f"actual={result['mean_qed']:.3f}, "
              f"uniq={result['uniqueness']:.1f}%, "
              f"valid={result['valid']}")

# Summary table
print("\n" + "="*70)
print("SUMMARY: Find the cfg_scale with best control + diversity balance")
print("="*70)
import pandas as pd
df_results = pd.DataFrame(results)
for target in test_targets:
    print(f"\n🎯 Target QED = {target}")
    subset = df_results[df_results['target_qed'] == target]
    print(subset[['cfg_scale', 'mean_qed', 'uniqueness']].to_string(index=False))

### 🛠️ Optional: CFG Rescale (Stabilize Logits Magnitude)

If high CFG scales cause collapse, try rescaling to match unconditional std:

In [ ]:
# Add CFG rescaling to p_sample_step (modify existing function)
# This goes in the p_sample_step function after computing logits

# BEFORE (in your current p_sample_step):
# logits = logits_u + cfg_scale * (logits_c - logits_u)

# AFTER (with rescaling):
# logits = logits_u + cfg_scale * (logits_c - logits_u)
# # Rescale to match unconditional std (prevents magnitude explosion)
# std_u = logits_u.std(dim=-1, keepdim=True)
# std_combined = logits.std(dim=-1, keepdim=True)
# logits = logits * (std_u / (std_combined + 1e-8))

print("⚠️ To enable CFG rescaling, modify p_sample_step function (cell 19):")
print("   Add 3 lines after 'logits = logits_u + cfg_scale * (logits_c - logits_u)'")
print("   See code above for exact implementation.")

In [ ]:
# ===========================
# Cell 5: Quick target sweep
# ===========================
for q in [0.2, 0.5, 0.85]:
    mean_qed, uniq, nvalid = eval_target_qed(q, sa_target=3.0, n=200, steps=300, temperature=0.7)
    print(f"Target QED={q}: meanQED={mean_qed}, uniq={uniq:.1f}%, valid={nvalid}")



In [ ]:
# ===========================
# Cell 6: HIGH QED run
# ===========================
print("\n" + "="*60)
print("Generating HIGH QED molecules (target QED~0.85, SA~3.0)")
print("="*60)

target_high_qed = {"qed": 0.85, "sa_score": 3.0}
smiles_high = sample_conditional(
    target_high_qed,
    n=100,
    steps=config.SAMPLE_STEPS,
    temperature=config.TEMPERATURE
)

qed_scores = []
for s in smiles_high:
    m = Chem.MolFromSmiles(s)
    if m:
        qed_scores.append(QED.qed(m))
qed_scores = np.array(qed_scores, dtype=np.float32) if qed_scores else np.array([], dtype=np.float32)

validity = (len(smiles_high) / 100) * 100
uniqueness = (len(set(smiles_high)) / len(smiles_high) * 100) if smiles_high else 0

print("\nResults:")
print(f"  Validity:   {validity:.1f}%")
print(f"  Uniqueness: {uniqueness:.1f}%")
if qed_scores.size:
    print(f"  Mean QED:   {qed_scores.mean():.3f}")
    print(f"  Median QED: {np.median(qed_scores):.3f}")
    print(f"  % QED > 0.7: {(qed_scores > 0.7).mean()*100:.1f}%")
    print(f"  % QED > 0.8: {(qed_scores > 0.8).mean()*100:.1f}%")
else:
    print("  QED stats:  N/A")

print("\nTop 5 molecules by QED:")
if qed_scores.size:
    pairs = sorted(zip(smiles_high, qed_scores.tolist()), key=lambda x: x[1], reverse=True)
    for i, (s, q) in enumerate(pairs[:5], 1):
        print(f"  {i}. QED={q:.3f}: {s[:80]}{'...' if len(s)>80 else ''}")
else:
    print("  N/A")


In [ ]:

# ===========================
# Cell 7: LOW SA run
# ===========================
print("\n" + "="*60)
print("Generating EASY-TO-SYNTHESIZE molecules (target SA~2.5, QED~0.7)")
print("="*60)

from rdkit.Chem import RDConfig
sys.path.append(os.path.join(RDConfig.RDContribDir, "SA_Score"))
import sascorer

target_low_sa = {"qed": 0.7, "sa_score": 2.5}
smiles_low_sa = sample_conditional(
    target_low_sa,
    n=100,
    steps=config.SAMPLE_STEPS,
    temperature=config.TEMPERATURE
)

sa_scores = []
for s in smiles_low_sa:
    m = Chem.MolFromSmiles(s)
    if m:
        sa_scores.append(sascorer.calculateScore(m))
sa_scores = np.array(sa_scores, dtype=np.float32) if sa_scores else np.array([], dtype=np.float32)

print("\nResults:")
print(f"  Validity: {len(smiles_low_sa)/100*100:.1f}%")
if sa_scores.size:
    print(f"  Mean SA Score: {sa_scores.mean():.2f}")
    print(f"  % with SA < 3.5: {(sa_scores < 3.5).mean()*100:.1f}%")
else:
    print("  SA stats: N/A")

print("\nTop 5 molecules by SA (easiest):")
if sa_scores.size:
    pairs = sorted(zip(smiles_low_sa, sa_scores.tolist()), key=lambda x: x[1])
    for i, (s, sa) in enumerate(pairs[:5], 1):
        print(f"  {i}. SA={sa:.2f}: {s[:80]}{'...' if len(s)>80 else ''}")
else:
    print("  N/A")


---
# Stage 3: Affinity-Conditioned Diffusion
---

Fine-tune the conditional model on molecules with valid binding affinity data (`pchembl_mean`).

**Why Stage 3?**
- Stage 2 uses 8 high-coverage features (>50% coverage)
- pchembl_mean has only ~21% coverage (42k/200k molecules)
- Fine-tuning on affinity subset allows explicit potency conditioning
- Transfer from Stage 2 provides strong structural priors

In [ ]:
# ==============================================================================
# STAGE 3: AFFINITY-CONDITIONED DIFFUSION
# ==============================================================================

print("="*60)
print("STAGE 3: AFFINITY-CONDITIONED DIFFUSION")
print("="*60)

# Filter molecules with valid pchembl_mean
df_aff = df[df["pchembl_mean"].notna()].copy().reset_index(drop=True)
print(f"\nAffinity subset: {len(df_aff):,} / {len(df):,} ({len(df_aff)/len(df)*100:.1f}%)")

# ==============================================================================
# FIX #2: SHUFFLE BEFORE SPLITTING (avoid temporal/ordering bias)
# ==============================================================================
df_aff = df_aff.sample(frac=1, random_state=42).reset_index(drop=True)
print("✅ Shuffled affinity subset with seed=42 (avoids ordering bias)")

# Stage 3 features = Stage 2 features + pchembl_mean
# Explicitly define to avoid any ambiguity
stage3_features = [
    'qed',          # Drug-likeness
    'sa_score',     # Synthetic accessibility
    'mw',           # Molecular weight
    'alogp',        # Lipophilicity
    'hbd',          # H-bond donors
    'hba',          # H-bond acceptors
    'psa',          # Polar surface area
    'num_rings',    # Ring count
    'pchembl_mean'  # NEW: Binding affinity
]
print(f"\nStage 3 features ({len(stage3_features)}): {stage3_features}")

# Verify all features are available
for feat in stage3_features:
    if feat in df_aff.columns:
        coverage = df_aff[feat].notna().sum() / len(df_aff) * 100
        print(f"  ✅ {feat}: {coverage:.1f}% coverage in affinity subset")
    else:
        print(f"  ❌ {feat}: NOT FOUND")

In [ ]:
# Create Stage 3 datasets
train_size_aff = int(0.9 * len(df_aff))
train_df_aff = df_aff.iloc[:train_size_aff].reset_index(drop=True)
val_df_aff = df_aff.iloc[train_size_aff:].reset_index(drop=True)

print(f"\nStage 3 splits:")
print(f"  Train: {len(train_df_aff):,} molecules")
print(f"  Val:   {len(val_df_aff):,} molecules")

# Create datasets with Stage 3 features
train_dataset_aff = MoleculeDataset(train_df_aff, condition_features=stage3_features)
val_dataset_aff = MoleculeDataset(val_df_aff, condition_features=stage3_features)

# Save normalization stats for Stage 3
aff_cond_stats = {
    'features': stage3_features,
    'mean': train_dataset_aff.cond_mean.tolist(),
    'std': train_dataset_aff.cond_std.tolist()
}
with open(os.path.join(config.MODEL_DIR, 'diffusion_affinity_condition_stats.json'), 'w') as f:
    json.dump(aff_cond_stats, f, indent=2)
print(f"\n✅ Saved Stage 3 normalization stats")

# Create dataloaders
train_loader_aff = DataLoader(
    train_dataset_aff,
    batch_size=config.BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)
val_loader_aff = DataLoader(
    val_dataset_aff,
    batch_size=config.BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

print(f"Train batches: {len(train_loader_aff)}")
print(f"Val batches: {len(val_loader_aff)}")

In [ ]:
# ==============================================================================
# FIX #1: CONDITION_DIM CONSISTENCY
# ==============================================================================
# condition_dim should be INDEPENDENT of n_conditions
# Stage 2 uses condition_dim=64, Stage 3 MUST use the same value
# Only n_conditions changes (8 → 9), not condition_dim

STAGE3_CONDITION_DIM = config.CONDITION_DIM  # Keep identical to Stage 2 (64)
print(f"Stage 3 condition_dim: {STAGE3_CONDITION_DIM} (same as Stage 2)")
print(f"Stage 3 n_conditions: {len(stage3_features)} (Stage 2 was 8)")

# Build Stage 3 model with expanded condition space
model_aff = DiffusionTransformer(
    vocab_size=VOCAB_SIZE,
    max_len=config.MAX_LEN,
    d_model=config.D_MODEL,
    n_heads=config.N_HEADS,
    n_layers=config.N_LAYERS,
    d_ff=config.D_FF,
    dropout=config.DROPOUT,
    condition_dim=STAGE3_CONDITION_DIM,  # FIX: Explicitly same as Stage 2
    n_conditions=len(stage3_features)     # Only this changes: 8 → 9
).to(device)

print(f"Stage 3 model parameters: {count_parameters(model_aff):,}")

# ==============================================================================
# FIX #3: SAFER WEIGHT TRANSFER WITH FULL REPORTING
# ==============================================================================

print("\n" + "="*60)
print("WEIGHT TRANSFER FROM STAGE 2")
print("="*60)

# Load Stage 2 checkpoint
stage2_ckpt = torch.load(os.path.join(config.MODEL_DIR, 'diffusion_cond_best.pt'))
state2 = stage2_ckpt['model_state_dict']

# Get Stage 3 state dict (this is a COPY, not a reference!)
state3 = model_aff.state_dict()

# Transfer compatible weights with detailed tracking
loaded_keys = []
skipped_keys = []
shape_mismatches = []

for k, v in state2.items():
    if k in state3:
        if state3[k].shape == v.shape:
            state3[k] = v  # Modify the copy
            loaded_keys.append(k)
        else:
            skipped_keys.append(k)
            shape_mismatches.append((k, v.shape, state3[k].shape))
    else:
        skipped_keys.append(k)

# Load with strict=False and report any issues
missing, unexpected = model_aff.load_state_dict(state3, strict=False)

print(f"\n✅ Transferred: {len(loaded_keys)} tensors")
print(f"⏭️  Skipped: {len(skipped_keys)} tensors")

if shape_mismatches:
    print(f"\n📐 Shape mismatches (expected for cond_emb):")
    for k, old_shape, new_shape in shape_mismatches:
        print(f"   {k}: {old_shape} → {new_shape}")

if missing:
    print(f"\n⚠️  Missing keys (not in loaded state): {missing}")
if unexpected:
    print(f"\n⚠️  Unexpected keys: {unexpected}")

print("\n💡 cond_emb.0 layer has different input dim (8 → 9), randomly initialized")
print("   All other layers (token_emb, pos_emb, time_emb, transformer layers) transferred")

In [ ]:
# ==============================================================================
# TRAIN STAGE 3 (AFFINITY-CONDITIONED)
# ==============================================================================

# Fine-tuning hyperparameters
AFF_EPOCHS = 20      # Fewer epochs for fine-tuning
AFF_LR = 1e-4        # Same learning rate (transfer learning)

optimizer_aff = AdamW(model_aff.parameters(), lr=AFF_LR, weight_decay=config.WEIGHT_DECAY)
total_steps_aff = len(train_loader_aff) * AFF_EPOCHS
scheduler_aff = CosineAnnealingLR(optimizer_aff, T_max=total_steps_aff)

print("="*60)
print("TRAINING STAGE 3: AFFINITY-CONDITIONED")
print("="*60)
print(f"Epochs: {AFF_EPOCHS}, LR: {AFF_LR}, Batch: {config.BATCH_SIZE}")

train_losses_aff = []
val_losses_aff = []
best_val_loss_aff = float('inf')

for epoch in range(AFF_EPOCHS):
    # Train
    train_loss = train_epoch(
        model_aff, train_loader_aff, optimizer_aff, scheduler_aff,
        schedule, VOCAB_SIZE, epoch, conditional=True
    )
    train_losses_aff.append(train_loss)

    # Validate
    val_loss = validate(model_aff, val_loader_aff, schedule, VOCAB_SIZE, conditional=True)
    val_losses_aff.append(val_loss)

    print(f"Epoch {epoch+1}/{AFF_EPOCHS} - Train: {train_loss:.4f}, Val: {val_loss:.4f}")

    # Save best model
    if val_loss < best_val_loss_aff:
        best_val_loss_aff = val_loss
        torch.save({
            'epoch': epoch,
            'model_state_dict': model_aff.state_dict(),
            'optimizer_state_dict': optimizer_aff.state_dict(),
            'train_loss': train_loss,
            'val_loss': val_loss,
            'config': vars(config),
            'condition_features': stage3_features
        }, os.path.join(config.MODEL_DIR, 'diffusion_affinity_best.pt'))
        print(f"  ✅ Saved best Stage 3 model (val_loss: {val_loss:.4f})")

    # Checkpoint every 5 epochs
    if (epoch + 1) % 5 == 0:
        torch.save({
            'epoch': epoch,
            'model_state_dict': model_aff.state_dict(),
            'train_losses': train_losses_aff,
            'val_losses': val_losses_aff,
        }, os.path.join(config.MODEL_DIR, f'diffusion_affinity_epoch{epoch+1}.pt'))

print("\n✅ Stage 3 training complete!")

## Sample from Stage 3 (Affinity-Conditioned)

In [ ]:
# Load best Stage 3 model
checkpoint_aff = torch.load(os.path.join(config.MODEL_DIR, 'diffusion_affinity_best.pt'))
model_aff.load_state_dict(checkpoint_aff['model_state_dict'])
print(f"Loaded best Stage 3 model from epoch {checkpoint_aff['epoch']+1}")

# Load Stage 3 condition stats
with open(os.path.join(config.MODEL_DIR, 'diffusion_affinity_condition_stats.json'), 'r') as f:
    aff_cond_stats = json.load(f)

print(f"\nStage 3 conditioning features: {aff_cond_stats['features']}")

In [ ]:
# ==============================================================================
# Generate HIGH AFFINITY molecules (pChEMBL > 8.0 = IC50 < 10 nM)
# ==============================================================================

print("\n" + "="*60)
print("Generating HIGH AFFINITY molecules (pChEMBL > 8.0)")
print("="*60)

num_samples = 100

# Target: High affinity + good drug-likeness + easy synthesis
target_high_affinity = {
    'qed': 0.80,           # Good drug-likeness
    'sa_score': 3.0,       # Easy to synthesize
    'mw': 400.0,           # Medium MW
    'alogp': 2.5,          # Moderate lipophilicity
    'hbd': 2,              # Few H-bond donors
    'hba': 5,              # Moderate H-bond acceptors
    'psa': 80.0,           # Good oral absorption
    'num_rings': 3,        # Moderate complexity
    'pchembl_mean': 8.5    # HIGH AFFINITY (IC50 ~3 nM)
}

cond_vec_aff = prepare_conditions(target_high_affinity, aff_cond_stats, device)
cond_batch_aff = cond_vec_aff.unsqueeze(0).expand(num_samples, -1)
cond_mask_aff = torch.ones(num_samples, dtype=torch.bool, device=device)

samples_high_aff = sample(
    model_aff, schedule,
    num_samples=num_samples,
    seq_len=config.MAX_LEN,
    vocab_size=VOCAB_SIZE,
    sample_steps=config.SAMPLE_STEPS,
    temperature=config.TEMPERATURE,
    cond=cond_batch_aff,
    cond_mask=cond_mask_aff,
    device=device
)

# Evaluate
generated_selfies_aff = [decode_selfies(s.cpu().tolist()) for s in samples_high_aff]
generated_smiles_aff = [selfies_to_smiles(s) for s in generated_selfies_aff]
valid_smiles_aff = [s for s in generated_smiles_aff if is_valid_molecule(s)]

qed_scores_aff = []
for smi in valid_smiles_aff:
    mol = Chem.MolFromSmiles(smi)
    if mol:
        qed_scores_aff.append(QED.qed(mol))

print(f"\nResults (High Affinity Conditioning):")
print(f"  Validity: {len(valid_smiles_aff)/len(generated_smiles_aff)*100:.1f}%")
print(f"  Uniqueness: {len(set(valid_smiles_aff))/len(valid_smiles_aff)*100:.1f}%" if valid_smiles_aff else "N/A")
print(f"  Mean QED: {np.mean(qed_scores_aff):.3f}" if qed_scores_aff else "N/A")

print(f"\nSample HIGH AFFINITY molecules:")
for i, smi in enumerate(valid_smiles_aff[:5]):
    mol = Chem.MolFromSmiles(smi)
    qed = QED.qed(mol) if mol else 0
    mw = Descriptors.MolWt(mol) if mol else 0
    print(f"  {i+1}. QED={qed:.3f}, MW={mw:.0f}: {smi[:50]}...")

In [ ]:
# ==============================================================================
# FIX #5: PROPER AFFINITY CONTROL EVALUATION
# ==============================================================================
# Compare LOW vs HIGH affinity generation with affinity predictor

print("\n" + "="*60)
print("COMPARISON: LOW vs HIGH Affinity Conditioning")
print("="*60)

# LOW affinity target (pChEMBL ~5.5 = IC50 ~3 μM)
target_low_affinity = target_high_affinity.copy()
target_low_affinity['pchembl_mean'] = 5.5  # LOW AFFINITY

cond_vec_low = prepare_conditions(target_low_affinity, aff_cond_stats, device)
cond_batch_low = cond_vec_low.unsqueeze(0).expand(num_samples, -1)

samples_low_aff = sample(
    model_aff, schedule,
    num_samples=num_samples,
    seq_len=config.MAX_LEN,
    vocab_size=VOCAB_SIZE,
    sample_steps=config.SAMPLE_STEPS,
    temperature=config.TEMPERATURE,
    cond=cond_batch_low,
    cond_mask=cond_mask_aff,
    device=device
)

# Evaluate LOW affinity samples
gen_selfies_low = [decode_selfies(s.cpu().tolist()) for s in samples_low_aff]
gen_smiles_low = [selfies_to_smiles(s) for s in gen_selfies_low]
valid_smiles_low = [s for s in gen_smiles_low if is_valid_molecule(s)]

# ==============================================================================
# TRAIN QUICK AFFINITY PREDICTOR (Ridge on ECFP4)
# ==============================================================================
print("\n--- Training Affinity Predictor for Evaluation ---")

from rdkit.Chem import AllChem
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score

def smiles_to_ecfp(smiles, radius=2, nbits=2048):
    """Convert SMILES to ECFP fingerprint."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return np.array(AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=nbits))

# Prepare training data from affinity subset
print("Preparing fingerprints for affinity predictor...")
X_aff, y_aff = [], []
for _, row in df_aff.sample(min(10000, len(df_aff)), random_state=42).iterrows():
    fp = smiles_to_ecfp(row['canonical_smiles'] if 'canonical_smiles' in df_aff.columns else row['smiles'])
    if fp is not None:
        X_aff.append(fp)
        y_aff.append(row['pchembl_mean'])

X_aff = np.array(X_aff)
y_aff = np.array(y_aff)

# Train Ridge regression
aff_predictor = Ridge(alpha=1.0)
aff_predictor.fit(X_aff, y_aff)

# Cross-validation score
cv_scores = cross_val_score(aff_predictor, X_aff, y_aff, cv=5, scoring='r2')
print(f"Affinity predictor R²: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")

# ==============================================================================
# PREDICT AFFINITY FOR GENERATED MOLECULES
# ==============================================================================
def predict_affinity(smiles_list):
    """Predict pChEMBL for list of SMILES."""
    predictions = []
    for smi in smiles_list:
        fp = smiles_to_ecfp(smi)
        if fp is not None:
            pred = aff_predictor.predict([fp])[0]
            predictions.append(pred)
    return predictions

# Predict for LOW and HIGH affinity samples
pred_low = predict_affinity(valid_smiles_low)
pred_high = predict_affinity(valid_smiles_aff)

# ==============================================================================
# COMPREHENSIVE COMPARISON TABLE
# ==============================================================================

def get_mean_mw(smiles_list):
    mws = []
    for smi in smiles_list:
        mol = Chem.MolFromSmiles(smi)
        if mol:
            mws.append(Descriptors.MolWt(mol))
    return np.mean(mws) if mws else 0

def get_mean_num_rings(smiles_list):
    rings = []
    for smi in smiles_list:
        mol = Chem.MolFromSmiles(smi)
        if mol:
            rings.append(Descriptors.RingCount(mol))
    return np.mean(rings) if rings else 0

def compute_novelty_vs_training(smiles_list, train_fps, threshold=0.75):
    """Compute % of molecules with Tanimoto < threshold vs training set."""
    from rdkit import DataStructs
    novel_count = 0
    for smi in smiles_list:
        fp = smiles_to_ecfp(smi)
        if fp is None:
            continue
        fp_bit = DataStructs.CreateFromBitString(''.join(map(str, fp)))
        max_sim = 0
        for train_fp in train_fps[:1000]:  # Sample for speed
            train_bit = DataStructs.CreateFromBitString(''.join(map(str, train_fp)))
            sim = DataStructs.TanimotoSimilarity(fp_bit, train_bit)
            max_sim = max(max_sim, sim)
        if max_sim < threshold:
            novel_count += 1
    return novel_count / len(smiles_list) * 100 if smiles_list else 0

# Compute scaffold novelty
train_scaffolds = set(df_aff['bemis_murcko_scaffold'].dropna().unique()) if 'bemis_murcko_scaffold' in df_aff.columns else set()

def get_scaffold_novelty(smiles_list, train_scaffolds):
    from rdkit.Chem.Scaffolds import MurckoScaffold
    novel = 0
    for smi in smiles_list:
        mol = Chem.MolFromSmiles(smi)
        if mol:
            try:
                scaffold = MurckoScaffold.MurckoScaffoldSmiles(mol=mol)
                if scaffold not in train_scaffolds:
                    novel += 1
            except:
                pass
    return novel / len(smiles_list) * 100 if smiles_list else 0

print(f"\n{'='*70}")
print(f"{'Metric':<30} {'LOW (pChEMBL=5.5)':<20} {'HIGH (pChEMBL=8.5)':<20}")
print(f"{'='*70}")
print(f"{'Target pChEMBL':<30} {'5.5':<20} {'8.5':<20}")
print(f"{'Validity':<30} {len(valid_smiles_low)/num_samples*100:.1f}%{'':<15} {len(valid_smiles_aff)/num_samples*100:.1f}%")
print(f"{'Uniqueness':<30} {len(set(valid_smiles_low))/max(1,len(valid_smiles_low))*100:.1f}%{'':<15} {len(set(valid_smiles_aff))/max(1,len(valid_smiles_aff))*100:.1f}%")
print(f"{'Mean MW':<30} {get_mean_mw(valid_smiles_low):.1f}{'':<15} {get_mean_mw(valid_smiles_aff):.1f}")
print(f"{'Mean Rings':<30} {get_mean_num_rings(valid_smiles_low):.2f}{'':<15} {get_mean_num_rings(valid_smiles_aff):.2f}")
print(f"{'='*70}")
print(f"{'PREDICTED pChEMBL (mean±std)':<30} {np.mean(pred_low):.2f}±{np.std(pred_low):.2f}{'':<10} {np.mean(pred_high):.2f}±{np.std(pred_high):.2f}")
print(f"{'Δ from target':<30} {np.mean(pred_low)-5.5:+.2f}{'':<15} {np.mean(pred_high)-8.5:+.2f}")
print(f"{'='*70}")

# Statistical significance
from scipy import stats
t_stat, p_value = stats.ttest_ind(pred_low, pred_high)
print(f"\n📊 Statistical Test (t-test): t={t_stat:.2f}, p={p_value:.2e}")
if p_value < 0.05:
    print(f"   ✅ Significant difference (p<0.05): Affinity conditioning WORKS!")
else:
    print(f"   ⚠️  Not significant (p≥0.05): Conditioning may need more epochs")

print("\n💡 Interpretation:")
print(f"   • High affinity condition produces molecules with ~{np.mean(pred_high)-np.mean(pred_low):.2f} higher predicted pChEMBL")
print(f"   • This demonstrates controllable affinity-conditioned generation")

## Training Curves

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Stage 1: Unconditional
axes[0].plot(train_losses_uncond, label='Train', alpha=0.8)
axes[0].plot(val_losses_uncond, label='Val', alpha=0.8)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Stage 1: Unconditional Diffusion')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Stage 2: Conditional
axes[1].plot(train_losses_cond, label='Train', alpha=0.8)
axes[1].plot(val_losses_cond, label='Val', alpha=0.8)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].set_title('Stage 2: Conditional Diffusion')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Stage 3: Affinity-Conditioned
axes[2].plot(train_losses_aff, label='Train', alpha=0.8, color='green')
axes[2].plot(val_losses_aff, label='Val', alpha=0.8, color='darkgreen')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Loss')
axes[2].set_title('Stage 3: Affinity-Conditioned Diffusion')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(config.MODEL_DIR, 'diffusion_training_curves_all_stages.png'), dpi=150)
plt.show()

print("✅ Training curves saved for all 3 stages")

## Summary

In [ ]:
print("="*60)
print("TRAINING SUMMARY")
print("="*60)

print("\n📊 HYPERPARAMETERS:")
print(f"  Model: Transformer ({config.N_LAYERS} layers, {config.D_MODEL} dim, {config.N_HEADS} heads)")
print(f"  Parameters: {count_parameters(model_uncond):,} (uncond), {count_parameters(model_cond):,} (cond), {count_parameters(model_aff):,} (aff)")
print(f"  Diffusion: {config.TIMESTEPS} timesteps, {config.SCHEDULE} schedule")
print(f"  Training: {config.EPOCHS} epochs (S1/S2), {AFF_EPOCHS} epochs (S3)")
print(f"  Sequence: {config.MAX_LEN} tokens, {VOCAB_SIZE} vocab")

print("\n📁 SAVED FILES:")
print(f"  ✅ {config.MODEL_DIR}/diffusion_uncond_best.pt (Stage 1)")
print(f"  ✅ {config.MODEL_DIR}/diffusion_cond_best.pt (Stage 2)")
print(f"  ✅ {config.MODEL_DIR}/diffusion_affinity_best.pt (Stage 3)")
print(f"  ✅ {config.MODEL_DIR}/condition_stats.json (Stage 2 normalization)")
print(f"  ✅ {config.MODEL_DIR}/diffusion_affinity_condition_stats.json (Stage 3 normalization)")

print("\n🎯 TRAINING STAGES:")
print("  Stage 1: Unconditional (200k molecules)")
print(f"           → Learn general drug-like patterns")
print(f"  Stage 2: Conditional ({len(available_features)} features, 200k molecules)")
print(f"           → {available_features}")
print(f"  Stage 3: Affinity-Conditioned ({len(stage3_features)} features, {len(df_aff):,} molecules)")
print(f"           → {stage3_features}")

print("\n✅ NEXT STEPS:")
print("  1. Use Stage 3 model to generate molecules with target affinity")
print("  2. Fine-tune with RL for multi-objective optimization")
print("  3. Compare with VAE Stage 3 (same conditioning scheme)")
print("  4. Validate top candidates with docking simulations")